<a href="https://colab.research.google.com/github/llyasukell/Segmentation_images_medicales/blob/main/unet_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
os.chdir('/content/drive/MyDrive/colab')
print(os.getcwd())

/content/drive/MyDrive/colab


In [ ]:
!ls data/heart/Task02_Heart/imagesTr | head
!ls data/heart/Task02_Heart/labelsTr | head

la_003_0000.nii.gz
la_004_0000.nii.gz
la_005_0000.nii.gz
la_007_0000.nii.gz
la_009_0000.nii.gz
la_010_0000.nii.gz
la_011_0000.nii.gz
la_014_0000.nii.gz
la_016_0000.nii.gz
la_017_0000.nii.gz
la_003.nii.gz
la_004.nii.gz
la_005.nii.gz
la_007.nii.gz
la_009.nii.gz
la_010.nii.gz
la_011.nii.gz
la_014.nii.gz
la_016.nii.gz
la_017.nii.gz


In [4]:
!pip install -q monai nibabel scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 38.6 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [ ]:
!python training/train_unet.py

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-04-19 22:35:58.012692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776638158.042999    9671 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776638158.052905    9671 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776638158.079178    9671 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776638158.079248    9671 computation_placer.cc:1

In [ ]:
!pip install nibabel scipy scikit-learn


In [ ]:
!python training/train_post.py

Found 20 prediction files.
Fold: fold_0 | Case: la_007.nii.gz
  Original Dice:      0.9289
  Post-process Dice:  0.9289
  No change.
--------------------------------------------------
Fold: fold_0 | Case: la_016.nii.gz
  Original Dice:      0.9490
  Post-process Dice:  0.9490
  No change.
--------------------------------------------------
Fold: fold_0 | Case: la_021.nii.gz
  Original Dice:      0.9356
  Post-process Dice:  0.9356
  No change.
--------------------------------------------------
Fold: fold_0 | Case: la_024.nii.gz
  Original Dice:      0.9293
  Post-process Dice:  0.9296
  Largest CC removed false positives; Dice improved.
--------------------------------------------------
Fold: fold_1 | Case: la_003.nii.gz
  Original Dice:      0.9532
  Post-process Dice:  0.9532
  No change.
--------------------------------------------------
Fold: fold_1 | Case: la_018.nii.gz
  Original Dice:      0.9218
  Post-process Dice:  0.9218
  No change.
------------------------------------------

In [ ]:
!pip install -q nibabel matplotlib numpy scipy

In [ ]:
!python readgz.py

PROJECT_ROOT: /content/drive/MyDrive/colab
orig_root: /content/drive/MyDrive/colab/results/nnUNet_results/Dataset002_Heart/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres
post_root: /content/drive/MyDrive/colab/results/nnUNet_results_postprocessed/Dataset002_Heart/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres
gt_root: /content/drive/MyDrive/colab/results/gt_segmentations
img_root: /content/drive/MyDrive/colab/data/heart/Task02_Heart/imagesTr
------------------------------------------------------------
MRI path:          /content/drive/MyDrive/colab/data/heart/Task02_Heart/imagesTr/la_020_0000.nii.gz
Original pred:     /content/drive/MyDrive/colab/results/nnUNet_results/Dataset002_Heart/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_1/validation/la_020.nii.gz
Postprocessed:     /content/drive/MyDrive/colab/results/nnUNet_results_postprocessed/Dataset002_Heart/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_1/validation/la_020.nii.gz
Ground truth:      /content/drive/MyD

In [21]:
import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.ndimage import label, binary_erosion

pred_root = Path("/content/drive/MyDrive/colab/results/nnUNet_results/Dataset002_Heart/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres")
gt_dir = Path("/content/drive/MyDrive/colab/data/heart/Task02_Heart/labelsTr")

DICE_REFERENCE_THRESHOLD = dice

def dice_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    inter = np.logical_and(pred, gt).sum()
    return 2 * inter / (pred.sum() + gt.sum() + 1e-8)

def extract_features(pred):
    pred_bin = pred > 0
    volume = pred_bin.sum()

    labeled, num_components = label(pred_bin)

    if num_components > 0:
        sizes = [(labeled == i).sum() for i in range(1, num_components + 1)]
        largest_component = max(sizes)
        largest_component_ratio = largest_component / (volume + 1e-8)

        small_voxels = sum(size for size in sizes if size < 100)
        small_component_ratio = small_voxels / (volume + 1e-8)
    else:
        largest_component_ratio = 0
        small_component_ratio = 0

    if volume > 0:
        eroded = binary_erosion(pred_bin)
        surface = pred_bin.astype(int) - eroded.astype(int)
        surface_ratio = surface.sum() / (volume + 1e-8)
    else:
        surface_ratio = 0

    return {
        "volume": volume,
        "num_components": num_components,
        "largest_component_ratio": largest_component_ratio,
        "small_component_ratio": small_component_ratio,
        "surface_ratio": surface_ratio
    }

pred_files = sorted(pred_root.glob("fold_*/**/*.nii.gz"))

rows = []

for pred_path in pred_files:
    case_id = pred_path.name.replace(".nii.gz", "")
    gt_path = gt_dir / f"{case_id}.nii.gz"

    if not gt_path.exists():
        print("GT not found:", case_id)
        continue

    fold_name = "unknown"
    for part in pred_path.parts:
        if part.startswith("fold_"):
            fold_name = part
            break

    pred = nib.load(pred_path).get_fdata()
    gt = nib.load(gt_path).get_fdata()

    features = extract_features(pred)
    dice = dice_score(pred, gt)

    rows.append({
        "fold": fold_name,
        "case_id": case_id,
        **features,
        "dice_reference": dice,
        "low_quality_reference": int(dice < DICE_REFERENCE_THRESHOLD),
        "prediction_path": str(pred_path),
        "ground_truth_path": str(gt_path)
    })

df = pd.DataFrame(rows)

DICE_REFERENCE_THRESHOLD = df["dice_reference"].mean()
df["low_quality_reference"] = (df["dice_reference"] < DICE_REFERENCE_THRESHOLD).astype(int)

if len(df) == 0:
    raise RuntimeError("No matching prediction and GT files found.")

df["volume_z"] = (df["volume"] - df["volume"].mean()) / (df["volume"].std() + 1e-8)

df["risk_score"] = (
    df["num_components"].rank(pct=True)
    + (1 - df["largest_component_ratio"])
    + df["volume_z"].abs().rank(pct=True)
    + df["surface_ratio"].rank(pct=True)
    + df["small_component_ratio"].rank(pct=True)
)

df_sorted = df.sort_values("risk_score", ascending=False)

top_k = max(1, int(0.2 * len(df_sorted)))
high_risk = df_sorted.head(top_k)
low_risk = df_sorted.tail(top_k)

print("Total cases:", len(df_sorted))
print("Reference threshold: Dice <", DICE_REFERENCE_THRESHOLD)
print("Low-quality reference cases:", df_sorted["low_quality_reference"].sum())

print("\nQA decision based on risk_score:")
print("Top 20% high-risk cases:", top_k)

print("\nEvaluation using Dice as reference:")
print("Mean Dice overall:", df_sorted["dice_reference"].mean())
print("Mean Dice high-risk group:", high_risk["dice_reference"].mean())
print("Mean Dice low-risk group:", low_risk["dice_reference"].mean())
print(
    "Low-quality cases captured in top 20%:",
    high_risk["low_quality_reference"].sum(),
    "/",
    df_sorted["low_quality_reference"].sum()
)

df_sorted.to_csv("qc_risk_score_evaluation.csv", index=False)

df_sorted[[
    "fold",
    "case_id",
    "risk_score",
    "dice_reference",
    "low_quality_reference",
    "volume",
    "num_components",
    "largest_component_ratio",
    "surface_ratio",
    "small_component_ratio",
    "volume_z"
]].head(20)

Total cases: 20
Reference threshold: Dice < 0.9271278969510295
Low-quality reference cases: 5

QA decision based on risk_score:
Top 20% high-risk cases: 4

Evaluation using Dice as reference:
Mean Dice overall: 0.9271278969510292
Mean Dice high-risk group: 0.9163429839398539
Mean Dice low-risk group: 0.9285892350374633
Low-quality cases captured in top 20%: 2 / 5


,fold,case_id,risk_score,dice_reference,low_quality_reference,volume,num_components,largest_component_ratio,surface_ratio,small_component_ratio,volume_z
6,fold_1,la_020,3.422772,0.886679,1,27490,2,0.977228,0.177737,0.000000,-1.864744
19,fold_4,la_010,3.126698,0.916929,1,40628,2,0.998302,0.159225,0.001698,-0.520041
14,fold_3,la_029,3.112876,0.932495,0,33862,2,0.987124,0.168980,0.000000,-1.212556
3,fold_0,la_024,2.675796,0.929269,0,48985,2,0.999204,0.144350,0.000796,0.335316
12,fold_3,la_017,2.400000,0.936373,0,35429,1,1.000000,0.163934,0.000000,-1.052170
5,fold_1,la_018,2.400000,0.921848,1,35775,1,1.000000,0.167100,0.000000,-1.016756
13,fold_3,la_022,2.250000,0.939139,0,34982,1,1.000000,0.152221,0.000000,-1.097922
8,fold_2,la_011,2.100000,0.939020,0,58403,1,1.000000,0.135952,0.000000,1.299269
11,fold_2,la_026,2.000000,0.930240,0,57803,1,1.000000,0.134543,0.000000,1.237858
2,fold_0,la_021,1.950000,0.935560,0,39762,1,1.000000,0.151124,0.000000,-0.608678
